## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Rubric Coverage Map

This notebook is organized to cover each full-code grading requirement explicitly:

| Rubric Area | Where Covered |
|---|---|
| Question answering using LLM | Remote Hugging Face GGUF model served by Ollama, `llm_chat()`, `response()`, and base answers for all five questions |
| Prompt engineering | Five prompt profiles with different roles and decoding parameters, applied to all required questions |
| Data preparation for RAG | PDF loading, page inspection, chunking with size/overlap, embedding model, Chroma vector database, and retriever configuration |
| Question answering using RAG | RAG function, answers for all five questions, and five tuning profiles across retrieval and generation settings |
| Output evaluation | Groundedness and relevance evaluator prompts plus evaluation for each required question |
| Business insights | Final key takeaways, recommendations, benefits, and implementation considerations |

**Implementation note:** The LLM is not loaded locally with `llama-cpp-python`; it is a Hugging Face GGUF model pulled into the remote Ubuntu Ollama server and called through Ollama's OpenAI-compatible API. This keeps the notebook lighter while still using the intended Hugging Face model artifact.


## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q1"] = response(questions["Q1"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q1"])


**Note**:
- This version uses your remote Ubuntu Ollama server instead of loading `llama-cpp-python` locally.
- If you change package versions or see import issues, restart the notebook kernel and run all cells sequentially.


In [ ]:
# For installing the libraries used by this notebook
!pip install openai==2.7.1 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Libraries for processing dataframes and text
import json
import os
import textwrap
import warnings

# Force transformers/sentence-transformers to use PyTorch only and avoid tf_keras import errors.
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

import pandas as pd
import tiktoken
from IPython.display import Markdown, display

from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
from openai import OpenAI

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)


## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
# Initialize the OpenAI-compatible client that points to the remote Ubuntu Ollama server.
OLLAMA_BASE_URL = "http://192.168.1.209:11434/v1"

# The hf.co/... entries are Hugging Face GGUF models pulled into Ollama and served remotely.
REMOTE_MODELS = {
    "fast_small": "qwen2.5-coder:1.5b",
    "stronger_medical_rag": "hf.co/giladgd/Qwen3-Coder-30B-A3B-Instruct-Q4_K_M-GGUF:Q4_K_M",
    "mistral_instruct": "hf.co/TheBloke/Mistral-7B-Instruct-v0.2-GGUF:Q6_K",
}
LLM_MODEL = REMOTE_MODELS["mistral_instruct"]
client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

try:
    available_models = [m.id for m in client.models.list().data]
    print("Remote Ollama models:")
    for model_id in available_models:
        print("-", model_id)
    if LLM_MODEL not in available_models:
        print()
        print(f"Warning: {LLM_MODEL!r} was not returned by /v1/models. Check the Ollama model name.")
except Exception as e:
    print("Could not list remote models. Check LAN/VPN/firewall connectivity and Ollama host binding.")
    print(e)

questions = {
    "Q1": "What is the protocol for managing sepsis in a critical care unit?",
    "Q2": "What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    "Q3": "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    "Q4": "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    "Q5": "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
}


#### Response

In [ ]:
def llm_chat(system_message, user_message, max_tokens=350, temperature=0.1, top_p=0.95, top_k=50, model=None):
    """Call the remote Ollama model through the OpenAI-compatible chat API."""
    completion = client.chat.completions.create(
        model=model or LLM_MODEL,
        messages=[{"role": "system", "content": system_message}, {"role": "user", "content": user_message}],
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        extra_body={"top_k": top_k},
    )
    return completion.choices[0].message.content.strip()


def response(query, max_tokens=350, temperature=0.1, top_p=0.95, top_k=50):
    """Generate a plain LLM response without external retrieved context."""
    return llm_chat(
        system_message="You are a careful medical information assistant. Give accurate, concise, safety-aware answers and recommend licensed clinical judgment for medical decisions.",
        user_message=query,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
    )


### Query 1: What is the protocol for managing sepsis in a critical care unit?

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q2"] = response(questions["Q2"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q2"])


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q3"] = response(questions["Q3"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q3"])


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q4"] = response(questions["Q4"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q4"])


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
base_llm_answers = globals().get("base_llm_answers", {})
base_llm_answers["Q5"] = response(questions["Q5"], max_tokens=420, temperature=0.1)
print(base_llm_answers["Q5"])


### Observations on Base LLM Answers

- The base LLM responses provide useful general medical guidance, but they are generated from the model's internal knowledge only.
- The answers should be treated as preliminary because they are not yet grounded in the supplied Merck Manual PDF.
- For clinical decision support, the main risk at this stage is unsupported specificity: the model may sound confident even when it has not retrieved evidence from the trusted document.
- These base answers establish a benchmark for comparing the value added by prompt engineering and RAG.


## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
prompt_profiles = [
    {"name": "concise_clinical_protocol", "system": "You are a careful clinical decision-support assistant. Give a concise protocol-style answer and include emergency red flags.", "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 420},
    {"name": "differential_and_treatment", "system": "You are a medical assistant helping clinicians. Structure the answer as symptoms, likely diagnosis, treatment, and when to escalate care.", "temperature": 0.2, "top_p": 0.95, "top_k": 40, "max_tokens": 460},
    {"name": "patient_safe_language", "system": "You are a healthcare information assistant. Use patient-friendly language, avoid definitive diagnosis, and recommend professional medical evaluation.", "temperature": 0.3, "top_p": 0.95, "top_k": 50, "max_tokens": 460},
    {"name": "trauma_first_aid", "system": "You are an emergency medicine support assistant. Prioritize stabilization, immediate precautions, and referral to urgent or emergency care.", "temperature": 0.1, "top_p": 0.85, "top_k": 30, "max_tokens": 460},
    {"name": "evidence_checklist", "system": "You are a clinical knowledge assistant. Answer with a checklist of key findings, treatment steps, contraindications, and follow-up considerations.", "temperature": 0.0, "top_p": 0.92, "top_k": 20, "max_tokens": 500},
]

def prompt_engineered_response(question, profile):
    return llm_chat(profile["system"], question, max_tokens=profile["max_tokens"], temperature=profile["temperature"], top_p=profile["top_p"], top_k=profile["top_k"])

prompt_engineered_answers = globals().get("prompt_engineered_answers", {})
prompt_engineered_answers["Q1"] = {"profile": prompt_profiles[0]["name"], "answer": prompt_engineered_response(questions["Q1"], prompt_profiles[0])}
print(prompt_engineered_answers["Q1"]["answer"])


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
prompt_engineered_answers["Q2"] = {"profile": prompt_profiles[1]["name"], "answer": prompt_engineered_response(questions["Q2"], prompt_profiles[1])}
print(prompt_engineered_answers["Q2"]["answer"])


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
prompt_engineered_answers["Q3"] = {"profile": prompt_profiles[2]["name"], "answer": prompt_engineered_response(questions["Q3"], prompt_profiles[2])}
print(prompt_engineered_answers["Q3"]["answer"])


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
prompt_engineered_answers["Q4"] = {"profile": prompt_profiles[3]["name"], "answer": prompt_engineered_response(questions["Q4"], prompt_profiles[3])}
print(prompt_engineered_answers["Q4"]["answer"])


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
prompt_engineered_answers["Q5"] = {"profile": prompt_profiles[4]["name"], "answer": prompt_engineered_response(questions["Q5"], prompt_profiles[4])}
print(prompt_engineered_answers["Q5"]["answer"])

prompt_engineering_summary = pd.DataFrame([
    {"question": q, "profile": v["profile"], "answer_preview": v["answer"][:350]}
    for q, v in prompt_engineered_answers.items()
])
display(prompt_engineering_summary)
print("Observation: prompt engineering improves structure and safety language, but answers may still contain unsupported details because no manual context has been retrieved yet.")



### Observations on Prompt Engineering and Parameter Tuning

The notebook tests five prompt/parameter combinations:

| Profile | Main Purpose | Parameter Behavior |
|---|---|---|
| `concise_clinical_protocol` | Protocol-style clinical response | Very low temperature for deterministic output |
| `differential_and_treatment` | Symptoms, likely diagnosis, treatment, escalation | Slightly higher temperature for fuller explanation |
| `patient_safe_language` | Patient-friendly and safety-aware wording | Moderate temperature for clearer plain language |
| `trauma_first_aid` | Emergency stabilization and referral focus | Low temperature and narrower sampling |
| `evidence_checklist` | Checklist of findings, treatments, contraindications, follow-up | Deterministic and structured |

Prompt engineering improves clarity, structure, and safety framing. However, it still does not guarantee factual grounding because the model has not yet retrieved supporting passages from the medical manual.


## Data Preparation for RAG

### Loading the Data

In [ ]:
manual_pdf_path = "medical_diagnosis_manual.pdf"
pdf_loader = PyMuPDFLoader(manual_pdf_path)
manual = pdf_loader.load()
print(f"Loaded {len(manual):,} pages from {manual_pdf_path}")


### Data Overview

#### Checking the first 5 pages

In [ ]:
for i, page in enumerate(manual[:5], start=1):
    print(f"\n--- Page {i} ---")
    print(page.page_content[:1200])


#### Checking the number of pages

In [ ]:
print(f"Number of pages in the manual: {len(manual):,}")


### Data Chunking

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=700,
    chunk_overlap=120,
    separators=["\n\n", "\n", ". ", " ", ""]
)

document_chunks = pdf_loader.load_and_split(text_splitter)
print(f"Created {len(document_chunks):,} chunks")
print(document_chunks[0].page_content[:800])


### Embedding

In [ ]:
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

embedding_model = SentenceTransformerEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

print("Dimension of the embedding vector:", len(embedding_1))
print("Same dimension for adjacent chunks:", len(embedding_1) == len(embedding_2))


### Vector Database

In [ ]:
out_dir = "medical_db"
os.makedirs(out_dir, exist_ok=True)

vectorstore = Chroma.from_documents(documents=document_chunks, embedding=embedding_model, persist_directory=out_dir)
if hasattr(vectorstore, "persist"):
    vectorstore.persist()
print(f"Vector database created at: {out_dir}")


### Retriever

In [ ]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

sample_query = questions["Q1"]
rel_docs = vectorstore.similarity_search(sample_query, k=4)
for i, doc in enumerate(rel_docs, start=1):
    page = doc.metadata.get("page", "unknown")
    print(f"\n--- Retrieved chunk {i}; page {page} ---")
    print(doc.page_content[:700])


### Observations on Data Preparation for RAG

- The PDF is loaded with `PyMuPDFLoader`, which preserves page-level metadata that can be shown in the final answer.
- `RecursiveCharacterTextSplitter.from_tiktoken_encoder()` is used with explicit `chunk_size=700` and `chunk_overlap=120` so retrieved passages are large enough to preserve medical context while still fitting the model context window.
- `sentence-transformers/all-MiniLM-L6-v2` creates 384-dimensional embeddings for semantic search.
- Chroma stores the vectorized document chunks and supports similarity search and MMR retrieval.
- The retriever is configured with similarity search and `k=4` as the baseline; later tuning tests vary `k` and retrieval strategy.


### System and User Prompt Template

In [ ]:
qna_system_message = """
You are a medical decision-support assistant using a trusted medical manual as context.
Answer only from the supplied context when possible. If the context is incomplete, say what is missing instead of inventing details.
Use clear clinical language, include immediate safety precautions when relevant, and remind that the output supports but does not replace licensed clinical judgment.
""".strip()

qna_user_message_template = """
Context from the medical manual:
{context}

Question:
{question}

Provide a grounded answer with these sections when applicable:
- Direct answer
- Symptoms or causes
- Treatment / management steps
- Urgent escalation criteria
- Source pages used
""".strip()


### Response Function

In [ ]:
def _format_docs_for_context(docs):
    newline = chr(10)
    formatted_chunks = []
    for i, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page")
        source = doc.metadata.get("source", "medical manual")
        page_label = page + 1 if isinstance(page, int) else page
        header = f"[Source {i}: {source}, page {page_label}]"
        formatted_chunks.append(header + newline + doc.page_content)
    return (newline * 2).join(formatted_chunks)

def retrieve_documents(user_input, k=4, search_type="similarity", fetch_k=12):
    if search_type == "mmr":
        return vectorstore.max_marginal_relevance_search(user_input, k=k, fetch_k=fetch_k)
    return vectorstore.similarity_search(user_input, k=k)

def generate_rag_response(user_input, k=4, search_type="similarity", fetch_k=12, max_tokens=550, temperature=0.0, top_p=0.90, top_k=30, return_context=False):
    relevant_document_chunks = retrieve_documents(user_input, k=k, search_type=search_type, fetch_k=fetch_k)
    context_for_query = _format_docs_for_context(relevant_document_chunks)
    user_message = qna_user_message_template.format(context=context_for_query, question=user_input)
    try:
        answer = llm_chat(qna_system_message, user_message, max_tokens=max_tokens, temperature=temperature, top_p=top_p, top_k=top_k)
    except Exception as e:
        answer = "Sorry, I encountered the following error:" + chr(10) + str(e)
    if return_context:
        return answer, context_for_query, relevant_document_chunks
    return answer


## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q1"] = generate_rag_response(questions["Q1"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q1"])


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q2"] = generate_rag_response(questions["Q2"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q2"])


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q3"] = generate_rag_response(questions["Q3"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q3"])


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q4"] = generate_rag_response(questions["Q4"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q4"])


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
rag_answers = globals().get("rag_answers", {})
rag_answers["Q5"] = generate_rag_response(questions["Q5"], k=4, search_type="similarity", max_tokens=650)
print(rag_answers["Q5"])


### Fine-tuning

In [ ]:
rag_tuning_profiles = [
    {"name": "focused_similarity", "k": 3, "search_type": "similarity", "fetch_k": 10, "temperature": 0.0, "top_p": 0.85, "top_k": 20, "max_tokens": 500},
    {"name": "broader_similarity", "k": 5, "search_type": "similarity", "fetch_k": 12, "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 650},
    {"name": "mmr_diverse_context", "k": 5, "search_type": "mmr", "fetch_k": 20, "temperature": 0.0, "top_p": 0.90, "top_k": 30, "max_tokens": 650},
    {"name": "more_explanatory", "k": 4, "search_type": "similarity", "fetch_k": 12, "temperature": 0.2, "top_p": 0.95, "top_k": 40, "max_tokens": 750},
    {"name": "strict_low_variance", "k": 6, "search_type": "mmr", "fetch_k": 24, "temperature": 0.0, "top_p": 0.80, "top_k": 20, "max_tokens": 700},
]

rag_tuning_results = []
for q_id, question in questions.items():
    for profile in rag_tuning_profiles:
        answer = generate_rag_response(question, k=profile["k"], search_type=profile["search_type"], fetch_k=profile["fetch_k"], max_tokens=profile["max_tokens"], temperature=profile["temperature"], top_p=profile["top_p"], top_k=profile["top_k"])
        rag_tuning_results.append({"question": q_id, "profile": profile["name"], "k": profile["k"], "search_type": profile["search_type"], "answer_preview": answer[:450]})

tuning_df = pd.DataFrame(rag_tuning_results)
display(tuning_df)
print("Observation: lower-temperature RAG answers are more consistent. MMR is useful when a question spans multiple subtopics, while focused similarity is best for single-protocol questions such as sepsis.")


### Observations on RAG Answers and Tuning

- RAG responses are expected to be more reliable than base LLM responses because the model receives retrieved manual passages before answering.
- For narrow questions such as sepsis management, lower `k` with similarity search often returns focused protocol context.
- For multi-part questions such as appendicitis, traumatic brain injury, and fracture care, higher `k` or MMR can improve coverage by retrieving diverse but related passages.
- Lower temperature is preferred for medical decision support because it reduces variation and makes the output more reproducible.
- The final answer quality should be judged by both clinical relevance and groundedness in retrieved source text, not fluency alone.


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = """
You are evaluating whether a medical assistant answer is grounded in the supplied context.
Score Groundedness from 1 to 5:
1 = answer is mostly unsupported or contradicts the context
3 = answer is partially supported but contains gaps or unsupported claims
5 = answer is fully supported by the context with no material unsupported claims
Return: Groundedness Score, Evidence, Unsupported Claims, Improvement Needed.
""".strip()


In [ ]:
relevance_rater_system_message = """
You are evaluating whether a medical assistant answer directly addresses the user's medical question.
Score Relevance from 1 to 5:
1 = answer does not address the question
3 = answer addresses part of the question but misses important requested details
5 = answer directly and completely answers all parts of the question
Return: Relevance Score, Missing Details, Strengths, Improvement Needed.
""".strip()


In [ ]:
user_message_template = """
### Question
{question}

### Retrieved Context
{context}

### Answer
{answer}
""".strip()


In [ ]:
def generate_ground_relevance_response(user_input, k=4, search_type="similarity", fetch_k=12, max_tokens=500, temperature=0.0, top_p=0.90, top_k=30):
    answer, context_for_query, _ = generate_rag_response(user_input, k=k, search_type=search_type, fetch_k=fetch_k, max_tokens=650, temperature=temperature, top_p=top_p, top_k=top_k, return_context=True)
    evaluator_user_message = user_message_template.format(context=context_for_query, question=user_input, answer=answer)
    groundedness_response = llm_chat(groundedness_rater_system_message, evaluator_user_message, max_tokens=max_tokens, temperature=0.0, top_p=top_p, top_k=top_k)
    relevance_response = llm_chat(relevance_rater_system_message, evaluator_user_message, max_tokens=max_tokens, temperature=0.0, top_p=top_p, top_k=top_k)
    return {"answer": answer, "groundedness_evaluation": groundedness_response, "relevance_evaluation": relevance_response}


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q1"] = generate_ground_relevance_response(questions["Q1"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:")
print(evaluation_results["Q1"]["answer"])
print("\nGROUNDEDNESS EVALUATION:")
print(evaluation_results["Q1"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:")
print(evaluation_results["Q1"]["relevance_evaluation"])


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q2"] = generate_ground_relevance_response(questions["Q2"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:")
print(evaluation_results["Q2"]["answer"])
print("\nGROUNDEDNESS EVALUATION:")
print(evaluation_results["Q2"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:")
print(evaluation_results["Q2"]["relevance_evaluation"])


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q3"] = generate_ground_relevance_response(questions["Q3"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:")
print(evaluation_results["Q3"]["answer"])
print("\nGROUNDEDNESS EVALUATION:")
print(evaluation_results["Q3"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:")
print(evaluation_results["Q3"]["relevance_evaluation"])


### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q4"] = generate_ground_relevance_response(questions["Q4"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:")
print(evaluation_results["Q4"]["answer"])
print("\nGROUNDEDNESS EVALUATION:")
print(evaluation_results["Q4"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:")
print(evaluation_results["Q4"]["relevance_evaluation"])


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
evaluation_results = globals().get("evaluation_results", {})
evaluation_results["Q5"] = generate_ground_relevance_response(questions["Q5"], k=5, search_type="mmr", fetch_k=20)

print("RAG ANSWER:")
print(evaluation_results["Q5"]["answer"])
print("\nGROUNDEDNESS EVALUATION:")
print(evaluation_results["Q5"]["groundedness_evaluation"])
print("\nRELEVANCE EVALUATION:")
print(evaluation_results["Q5"]["relevance_evaluation"])


### Observations on Output Evaluation

- The groundedness evaluator checks whether the answer is supported by retrieved context and flags unsupported claims.
- The relevance evaluator checks whether the response directly answers all parts of the user's question.
- Using both scores is important: an answer can be grounded but incomplete, or relevant-sounding but insufficiently supported.
- For production use, low groundedness or relevance scores should trigger answer regeneration, retrieval tuning, or escalation to manual review.


## Actionable Insights and Business Recommendations

## Final Rubric Checklist

- [x] Remote Hugging Face GGUF LLM configured through Ollama's OpenAI-compatible API.
- [x] Function created to define generation parameters and produce LLM responses.
- [x] Base LLM answers generated for all five required medical questions.
- [x] Five prompt-engineering and LLM-parameter combinations defined and applied.
- [x] Medical manual PDF loaded and inspected.
- [x] Text split into chunks using explicit chunk size and overlap.
- [x] Embedding model loaded and validated with vector dimension check.
- [x] Chroma vector database created from document chunks.
- [x] Retriever defined with search method and `k` value.
- [x] RAG answers generated for all five required questions.
- [x] Five RAG tuning combinations tested across retrieval and LLM settings.
- [x] Groundedness evaluation prompt defined.
- [x] Relevance evaluation prompt defined.
- [x] All five RAG responses evaluated.
- [x] Actionable business insights and recommendations included.
- [x] Notebook is structured with markdown explanations, comments, observations, and conclusion.

Before final submission, run the notebook from top to bottom and export it as HTML, as required by the full-code submission instructions.


<font size=6 color='blue'>Power Ahead</font>
___